# YOLOv8n + DINOv2 on Pascal VOC

Portable Kaggle/Google Colab notebook for a controlled pretrained-YOLO baseline versus DINOv2 global-context fusion experiment. The notebook contains the custom fusion module and configuration, so it can run from the notebook alone; cloning a repository is optional for training and required only for the optional Git commit/push section.

## What the fusion changes

- `ConvDummy` carries the original RGB tensor through the YOLO graph without modifying it.
- A frozen DINOv2 ViT-S/14 processes that original image and returns its global class-token embedding.
- A trainable `1x1` projection converts the 384-value embedding into a small three-channel context map, broadcast to YOLO's deepest feature-map resolution.
- The context map is concatenated with YOLO's deepest spatial feature. The projector and YOLO detector train; the DINOv2 backbone stays frozen.
- Pretrained YOLO weights are transferred to matching layers. The two expanded C2f input convolutions retain their pretrained weights and receive small nonzero weights for the new context channels.

This experiment tests whether that added scene context helps detection; it does not assume an improvement. The notebook uses Ultralytics' Pascal VOC config (16,551 train images and 4,952 VOC2007 test images used as validation). Do not use that same validation split as an independent final test after tuning. Dataset download is about 2.8 GB. Start with the smoke test, then run the full comparison on a GPU runtime.

In [ ]:
# 1. Runtime and Environment Setup (Colab/Kaggle)
import os
import platform
import random
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

# Mount Drive only when you choose to keep outputs in Colab Drive.
MOUNT_GOOGLE_DRIVE = False
if IS_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

print({
    "platform": "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else platform.platform(),
    "python": sys.version.split()[0],
    "cwd": str(Path.cwd()),
    "seed": SEED,
})


In [ ]:
# 2. Clone Repository and Select Working Branch
# Set this to your HTTPS Git URL if you want the later push cell to work.
REPO_URL = ""
BASE_BRANCH = "main"
WORK_BRANCH = "feature/yolov8-dinov2-voc"
REPO_DIR = None

if REPO_URL.strip():
    REPO_DIR = Path.cwd() / "project_repo"
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if status:
        raise RuntimeError("Repository has uncommitted changes; refusing to switch branches.")

    branches = subprocess.run(
        ["git", "-C", str(REPO_DIR), "branch", "--list", WORK_BRANCH],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if branches:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", WORK_BRANCH], check=True)
    else:
        subprocess.run([
            "git", "-C", str(REPO_DIR), "checkout", "-b", WORK_BRANCH,
            f"origin/{BASE_BRANCH}",
        ], check=True)
    print("Training is self-contained; repository checkout:", REPO_DIR)
else:
    print("No repository URL set. Training will run from the notebook; Git push stays disabled.")


In [ ]:
# 3. Install Dependencies and Validate GPU/Library Versions
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.142"
], check=True)

import importlib.util
import numpy as np
import torch
import torchvision
from importlib.metadata import version

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except TypeError:
    torch.use_deterministic_algorithms(True)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Ultralytics:", version("ultralytics"))
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("TensorFlow:", version("tensorflow") if importlib.util.find_spec("tensorflow") else "not installed (not used)")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=False)
else:
    raise RuntimeError("Enable a GPU accelerator in Kaggle or Colab before training.")


In [ ]:
# 4. Project Configuration for Portable Notebook Runs
from pathlib import Path

RUNTIME_ROOT = Path("/kaggle/working" if IS_KAGGLE else "/content" if IS_COLAB else Path.cwd())
WORK_DIR = RUNTIME_ROOT / "yolov8_dinov2_voc"
SOURCE_DIR = WORK_DIR / "fusion_source"
ARTIFACT_DIR = (
    Path("/content/drive/MyDrive/yolov8_dinov2_artifacts")
    if IS_COLAB and MOUNT_GOOGLE_DRIVE
    else RUNTIME_ROOT / "yolov8_dinov2_artifacts"
)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_YAML = WORK_DIR / "yolov8-dino-voc.yaml"

DATA = "VOC.yaml"  # Ultralytics downloads VOC2007/2012 on the first train/val call.
IMAGE_SIZE = 640
BATCH_SIZE = 2  # Lower to 1 if the selected GPU runs out of memory.
EPOCHS = 10
SMOKE_EPOCHS = 1
SMOKE_FRACTION = 0.01
WORKERS = 2
DEVICE = 0

print("Work directory:", WORK_DIR)
print("Artifacts:", ARTIFACT_DIR)
print("Dataset:", DATA, "(about 2.8 GB on first use)")
print("Train settings:", {"epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH_SIZE, "device": DEVICE})


## 5. Load the Baseline Pipeline and Fusion Modules

The baseline remains standard pretrained YOLOv8n. The fused graph adds a raw-image bypass (`ConvDummy`) and a DINOv2 branch, then joins the projected DINO context with the deepest YOLO feature before the neck/head. With YOLOv8n at 640px, the global DINO embedding has 384 values and the broadcast context has shape `(batch, 3, 20, 20)`; the image pyramid still feeds three detection scales.

## 6. Implement Fusion Code Changes

The next cell writes `dino_modules.py` into the runtime workspace and imports it under a stable module name so Ultralytics checkpoints can be reloaded during the same notebook session. Compared with the earlier experiment, it uses ImageNet RGB normalization and a small nonzero projector initialization. The following cell writes a nano-scale model YAML with 20 VOC classes.

In [ ]:
# 5-6. Define and Register the Fusion Modules
import importlib.util
import sys

MODULE_SOURCE = r'''from __future__ import annotations

import contextlib
import os
import warnings

import torch
from torch import nn
from torchvision import transforms as T


class ConvDummy(nn.Module):
    """Pass the original RGB input unchanged through the YOLO graph."""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x


class DINOv2(nn.Module):
    """Frozen global DINOv2 embedding with a trainable spatial projector."""
    def __init__(self, output_channels=3):
        super().__init__()
        with warnings.catch_warnings(), open(os.devnull, "w", encoding="utf-8") as sink:
            warnings.simplefilter("ignore")
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
        self.model.eval()
        self.model.requires_grad_(False)
        self.projector = nn.Conv2d(384, output_channels, kernel_size=1)
        nn.init.normal_(self.projector.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.projector.bias)
        self.transform = T.Compose([
            T.Resize((518, 518), antialias=True),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def train(self, mode=True):
        super().train(mode)
        self.model.eval()
        return self

    def forward(self, input_tensor):
        with torch.no_grad():
            embedding = self.model(self.transform(input_tensor))
            embedding = embedding.unsqueeze(-1).unsqueeze(-1)
        height = max(1, input_tensor.shape[2] // 32)
        width = max(1, input_tensor.shape[3] // 32)
        projected = self.projector(embedding)
        return torch.nn.functional.interpolate(
            projected, size=(height, width), mode="bilinear", align_corners=False
        )


def register_modules():
    from ultralytics.nn import tasks
    tasks.ConvDummy = ConvDummy
    tasks.DINOv2 = DINOv2
'''

MODULE_FILE = SOURCE_DIR / "dino_modules.py"
MODULE_FILE.write_text(MODULE_SOURCE, encoding="utf-8")
spec = importlib.util.spec_from_file_location("dino_modules", MODULE_FILE)
dino_modules = importlib.util.module_from_spec(spec)
sys.modules["dino_modules"] = dino_modules
spec.loader.exec_module(dino_modules)
dino_modules.register_modules()
print("Custom modules registered from:", MODULE_FILE)


In [ ]:
# 6. Generate the YOLOv8n + DINOv2 Architecture YAML
MODEL_YAML.parent.mkdir(parents=True, exist_ok=True)
MODEL_YAML.write_text('''
nc: 20
scale: n
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, ConvDummy, []]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [0, 1, DINOv2, [3]]
head:
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24], 1, Detect, [nc]]
''', encoding="utf-8")
print(MODEL_YAML.read_text(encoding="utf-8"))

### What changed versus plain YOLO

The custom YAML inserts two backbone layers, so downstream layer indices shift by two. The builder maps compatible YOLOv8n tensors back to their original layer indices. Two C2f input convolutions become three channels wider; the code copies the old filters and initializes only the added context-channel filters. Their context positions differ because one concat puts DINO first, while the deepest concat inserts DINO after the downsampled YOLO feature. Shape checks guard every transfer.

In [ ]:
# 6. Transfer Pretrained YOLOv8n Weights into the Fused Graph
from ultralytics import YOLO


def build_fused_model():
    dino_modules.register_modules()
    fused = YOLO(str(MODEL_YAML), verbose=False)
    pretrained = YOLO("yolov8n.pt", verbose=False).model.state_dict()
    fused_state = fused.model.state_dict()
    transferred = 0

    for key, value in fused_state.items():
        parts = key.split(".", 2)
        if len(parts) != 3 or parts[0] != "model" or not parts[1].isdigit():
            continue
        layer_index = int(parts[1])
        if 1 <= layer_index <= 10:
            source_index = layer_index - 1
        elif 13 <= layer_index <= 25:
            source_index = layer_index - 3
        else:
            continue
        source_value = pretrained.get(f"model.{source_index}.{parts[2]}")
        if source_value is None:
            continue
        if source_value.shape == value.shape:
            fused_state[key] = source_value
            transferred += 1
        elif (
            value.ndim == 4
            and source_value.ndim == 4
            and value.shape[0] == source_value.shape[0]
            and value.shape[1] == source_value.shape[1] + 3
            and value.shape[2:] == source_value.shape[2:]
        ):
            padded = torch.zeros_like(value)
            context_index = 0 if layer_index == 15 else source_value.shape[1] // 3
            padded[:, :context_index] = source_value[:, :context_index]
            padded[:, context_index + 3:] = source_value[:, context_index:]
            torch.nn.init.normal_(
                padded[:, context_index:context_index + 3], mean=0.0, std=0.01
            )
            fused_state[key] = padded
            transferred += 1

    fused.model.load_state_dict(fused_state)
    print(f"Transferred {transferred} compatible YOLOv8n tensors.")

    def freeze_dino_encoder(trainer):
        encoder = trainer.model.model[11].model
        encoder.requires_grad_(False)
        encoder.eval()

    def report_adapter_norm(trainer):
        projector = trainer.model.model[11].projector
        print(
            f"DINO adapter epoch {trainer.epoch + 1}: "
            f"weight_norm={float(projector.weight.detach().norm()):.6g}, "
            f"bias_norm={float(projector.bias.detach().norm()):.6g}"
        )

    fused.add_callback("on_train_start", freeze_dino_encoder)
    fused.add_callback("on_train_epoch_end", report_adapter_norm)
    return fused


## 7. Validate Fusion Outputs with Quick Unit Checks

This cell downloads the pretrained checkpoints if needed, checks that the DINO context and detector predictions have the expected dimensions, rejects non-finite values, then backpropagates one real YOLO detection loss and verifies that the projector receives gradients and changes after an optimizer step. It is a fast gate before a dataset-scale run.

In [ ]:
# 7. Forward, Shape, NaN, and Gradient Checks
from ultralytics.cfg import get_cfg

fused_check = build_fused_model()
fused_check.model.to(f"cuda:{DEVICE}").eval()
context_seen = {}
hook = fused_check.model.model[11].register_forward_hook(
    lambda module, inputs, output: context_seen.update(shape=tuple(output.shape))
)
with torch.inference_mode():
    prediction = fused_check.model(torch.zeros(1, 3, 320, 320, device=f"cuda:{DEVICE}"))
hook.remove()

pred_tensor = prediction[0]
assert context_seen["shape"] == (1, 3, 10, 10), context_seen
assert torch.isfinite(pred_tensor).all()
print("DINO context shape:", context_seen["shape"])
print("YOLO prediction shape:", tuple(pred_tensor.shape))

training_model = fused_check.model.train()
training_model.args = get_cfg()
training_model.criterion = training_model.init_criterion()
synthetic_batch = {
    "img": torch.rand(1, 3, 320, 320, device=f"cuda:{DEVICE}"),
    "batch_idx": torch.tensor([0.0], device=f"cuda:{DEVICE}"),
    "cls": torch.tensor([[0.0]], device=f"cuda:{DEVICE}"),
    "bboxes": torch.tensor([[0.5, 0.5, 0.2, 0.2]], device=f"cuda:{DEVICE}"),
}
projector = training_model.model[11].projector
initial_projector = projector.weight.detach().clone()
loss_parts, detached_losses = training_model.loss(synthetic_batch)
loss_parts.sum().backward()
projector_grad = projector.weight.grad
assert projector_grad is not None and torch.isfinite(projector_grad).all()
assert projector_grad.norm() > 0, "DINO projector received no detection-loss gradient"
optimizer = torch.optim.AdamW(
    [parameter for parameter in training_model.parameters() if parameter.requires_grad],
    lr=1e-4,
)
optimizer.step()
projector_delta = (projector.weight.detach() - initial_projector).norm()
assert projector_delta > 0, "Optimizer step did not update DINO projector"
print("Loss components:", {key: float(value) for key, value in detached_losses.items()})
print("Projector gradient norm:", float(projector_grad.norm()))
print("Projector update norm:", float(projector_delta))
del fused_check, training_model, optimizer


## 8. Run a Short Training Smoke Test

This one-epoch run uses 1% of VOC training images and skips validation to verify the end-to-end loader, loss, optimizer, checkpoint writing, and adapter-norm callback. The full VOC download happens on first use. If this smoke test fails or the adapter norm stays unchanged, stop here and debug before the full run.

In [ ]:
# 8. One-Epoch Fusion Smoke Test on a Fixed VOC Fraction
smoke_model = build_fused_model()
smoke_result = smoke_model.train(
    data=DATA,
    epochs=SMOKE_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    fraction=SMOKE_FRACTION,
    val=False,
    amp=False,
    seed=SEED,
    deterministic=True,
    plots=False,
    project=str(ARTIFACT_DIR / "smoke"),
    name="voc_fused_smoke",
)
print("Smoke run finished. Check the epoch log above for a changing DINO adapter norm.")
del smoke_model, smoke_result


### Full Matched Training Run

Run this only after the smoke test succeeds. Both models use the same VOC YAML, full training split, validation split, image size, batch, optimizer, epoch count, seed, and FP32 setting. The built-in VOC configuration uses VOC2007 test images as its validation split; treat these as the one comparison holdout and avoid repeated hyperparameter tuning against them.

In [ ]:
# 8. Train Baseline and Fused Model with Matched Settings
baseline_model = YOLO("yolov8n.pt", verbose=False)
baseline_result = baseline_model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    amp=False,
    optimizer="AdamW",
    lr0=0.001,
    close_mosaic=2,
    seed=SEED,
    deterministic=True,
    cls_remap=True,
    plots=True,
    project=str(ARTIFACT_DIR / "training"),
    name="voc_yolov8n_baseline",
)

fused_model = build_fused_model()
fused_result = fused_model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    amp=False,
    optimizer="AdamW",
    lr0=0.001,
    close_mosaic=2,
    seed=SEED,
    deterministic=True,
    cls_remap=True,
    pretrained=False,
    plots=True,
    project=str(ARTIFACT_DIR / "training"),
    name="voc_yolov8n_dinov2_fusion",
)

BASELINE_WEIGHTS = Path(baseline_result.save_dir) / "weights" / "best.pt"
FUSED_WEIGHTS = Path(fused_result.save_dir) / "weights" / "best.pt"
assert BASELINE_WEIGHTS.exists(), BASELINE_WEIGHTS
assert FUSED_WEIGHTS.exists(), FUSED_WEIGHTS
print("Baseline checkpoint:", BASELINE_WEIGHTS)
print("Fused checkpoint:", FUSED_WEIGHTS)


## 9. Evaluate, Save Artifacts, and Export for Reuse

Both checkpoints are reloaded and evaluated with the same VOC validation split and image size. The notebook writes JSON and CSV metrics, the generated model/module source, Ultralytics run plots, and a visual prediction into `ARTIFACT_DIR`. These files survive Kaggle/Colab only if saved under the shown working directory or mounted Drive.

In [ ]:
# 9. Evaluate at the Same Resolution and Save Comparison Artifacts
import csv
import json
import shutil
import cv2
from ultralytics.data.utils import check_det_dataset

baseline_eval_model = YOLO(str(BASELINE_WEIGHTS), verbose=False)
fused_eval_model = YOLO(str(FUSED_WEIGHTS), verbose=False)
baseline_metrics = baseline_eval_model.val(
    data=DATA, imgsz=IMAGE_SIZE, device=DEVICE, plots=True,
    project=str(ARTIFACT_DIR / "evaluation"), name="voc_baseline",
)
fused_metrics = fused_eval_model.val(
    data=DATA, imgsz=IMAGE_SIZE, device=DEVICE, plots=True,
    project=str(ARTIFACT_DIR / "evaluation"), name="voc_fusion",
)

comparison = [
    {
        "model": "YOLOv8n baseline",
        "precision": float(baseline_metrics.box.mp),
        "recall": float(baseline_metrics.box.mr),
        "map50": float(baseline_metrics.box.map50),
        "map50_95": float(baseline_metrics.box.map),
        "inference_ms_per_image": float(baseline_metrics.speed.get("inference", float("nan"))),
    },
    {
        "model": "YOLOv8n + DINOv2 fusion",
        "precision": float(fused_metrics.box.mp),
        "recall": float(fused_metrics.box.mr),
        "map50": float(fused_metrics.box.map50),
        "map50_95": float(fused_metrics.box.map),
        "inference_ms_per_image": float(fused_metrics.speed.get("inference", float("nan"))),
    },
]

results_json = ARTIFACT_DIR / "voc_comparison.json"
results_csv = ARTIFACT_DIR / "voc_comparison.csv"
results_json.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
with results_csv.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)
shutil.copy2(MODEL_YAML, ARTIFACT_DIR / MODEL_YAML.name)
shutil.copy2(MODULE_FILE, ARTIFACT_DIR / MODULE_FILE.name)

print("\nMatched validation results")
print("Model | P | R | mAP50 | mAP50-95 | inference ms/img")
for row in comparison:
    print(
        f"{row['model']} | {row['precision']:.4f} | {row['recall']:.4f} | "
        f"{row['map50']:.4f} | {row['map50_95']:.4f} | "
        f"{row['inference_ms_per_image']:.2f}"
    )
print("Saved:", results_json, results_csv)

# Export one validation image with baseline and fused detections side by side.
dataset_info = check_det_dataset(DATA)
val_entries = dataset_info["val"]
if isinstance(val_entries, (str, Path)):
    val_entries = [val_entries]
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
image_path = None
for entry in val_entries:
    candidate = Path(entry)
    if candidate.is_file() and candidate.suffix.lower() == ".txt":
        listed = [Path(line.strip()) for line in candidate.read_text(encoding="utf-8").splitlines() if line.strip()]
        image_path = next((item for item in listed if item.exists()), None)
    elif candidate.is_dir():
        image_path = next((item for item in candidate.rglob("*") if item.suffix.lower() in image_extensions), None)
    if image_path is not None:
        break
if image_path is not None:
    baseline_prediction = baseline_eval_model.predict(str(image_path), imgsz=IMAGE_SIZE, device=DEVICE, verbose=False)[0].plot()
    fused_prediction = fused_eval_model.predict(str(image_path), imgsz=IMAGE_SIZE, device=DEVICE, verbose=False)[0].plot()
    side_by_side = cv2.hconcat([baseline_prediction, fused_prediction])
    prediction_path = ARTIFACT_DIR / "voc_baseline_vs_fusion.jpg"
    cv2.imwrite(str(prediction_path), side_by_side)
    print("Prediction comparison:", prediction_path)
else:
    print("Could not locate a validation image for the optional side-by-side preview.")


## 10. Optional Commit and Push from the Notebook

Set `REPO_URL` in Section 2 and provide a GitHub personal access token only when prompted in the live runtime. The token is read with `getpass`, passed only to the child Git process through a temporary askpass helper, and never written into the notebook or Git remote URL. Review the staged file list before committing. This cell does not force-push.

In [ ]:
# 10. Commit and Push Only the Notebook-Generated Source/Docs
import getpass
import tempfile

if REPO_DIR is None or not (REPO_DIR / ".git").exists():
    print("Push skipped: set REPO_URL in Section 2 and rerun that cell first.")
else:
    notebook_name = "YOLOv8_DINOv2_VOC_Kaggle_Colab.ipynb"
    notebook_source = Path.cwd() / notebook_name
    if not notebook_source.exists():
        candidates = list(Path("/kaggle/input").glob(f"**/{notebook_name}")) if IS_KAGGLE else []
        notebook_source = candidates[0] if candidates else notebook_source
    if notebook_source.exists():
        shutil.copy2(notebook_source, REPO_DIR / notebook_name)

    scripts_dir = REPO_DIR / "scripts"
    configs_dir = REPO_DIR / "configs"
    scripts_dir.mkdir(exist_ok=True)
    configs_dir.mkdir(exist_ok=True)
    shutil.copy2(MODULE_FILE, scripts_dir / "dino_modules.py")
    shutil.copy2(MODEL_YAML, configs_dir / "yolov8-dino-voc.yaml")

    files_to_stage = [
        "YOLOv8_DINOv2_VOC_Kaggle_Colab.ipynb",
        "scripts/dino_modules.py",
        "configs/yolov8-dino-voc.yaml",
    ]
    existing_files = [item for item in files_to_stage if (REPO_DIR / item).exists()]
    subprocess.run(["git", "-C", str(REPO_DIR), "add", *existing_files], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "diff", "--cached", "--stat"], check=True)

    approve = input("Commit and push these staged files? Type YES to continue: ")
    if approve == "YES":
        git_name = input("Git author name: ").strip()
        git_email = input("Git author email: ").strip()
        username = input("GitHub username: ").strip()
        token = getpass.getpass("GitHub token (input hidden; never saved): ")
        subprocess.run(["git", "-C", str(REPO_DIR), "config", "user.name", git_name], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "config", "user.email", git_email], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "commit", "-m", "Add portable YOLOv8 DINOv2 VOC notebook"], check=True)
        askpass = Path(tempfile.gettempdir()) / "git-askpass-yolo-dino.sh"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in *Username*) printf "%s\\n" "$GIT_USERNAME" ;; *) printf "%s\\n" "$GIT_TOKEN" ;; esac\n',
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env = os.environ.copy()
        git_env.update({
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "GIT_USERNAME": username,
            "GIT_TOKEN": token,
        })
        try:
            subprocess.run(["git", "-C", str(REPO_DIR), "push", "-u", "origin", WORK_BRANCH], env=git_env, check=True)
        finally:
            token = None
            git_env.pop("GIT_TOKEN", None)
            askpass.unlink(missing_ok=True)
        print("Pushed branch:", WORK_BRANCH)
    else:
        print("Commit/push cancelled; staged changes are still available in this runtime.")


## 11. Kaggle/Colab Import and Re-run Checklist

Use GPU accelerator, run cells from top to bottom, allow the first VOC/DINO downloads to finish, confirm the smoke test changes the adapter norm, then start the full matched run. In Kaggle, download `ARTIFACT_DIR` as a notebook output or attach it as a dataset. In Colab, enable `MOUNT_GOOGLE_DRIVE` to persist outputs in Drive.

In [ ]:
# 11. Final Portability and Rerun Checks
from ultralytics import YOLO

assert torch.cuda.is_available(), "Select a GPU accelerator."
assert MODEL_YAML.exists() and MODULE_FILE.exists()
assert DEVICE == 0
assert IMAGE_SIZE % 32 == 0
assert 0 < SMOKE_FRACTION <= 1
assert ARTIFACT_DIR.exists()

if "BASELINE_WEIGHTS" in globals() and "FUSED_WEIGHTS" in globals():
    assert BASELINE_WEIGHTS.exists()
    assert FUSED_WEIGHTS.exists()
    print("Both trained checkpoints are available.")
else:
    print("Training has not run yet; checkpoint check deferred.")

print("Notebook environment checks passed.")
print("Next: run the short smoke test, inspect adapter norm, then run full training.")
